# Urhobo TTS Engine — Phase 4: Forced Alignment (Kaggle GPU)

This notebook runs **Meta MMS CTC Forced Alignment** on the complete Book of Genesis (all 50 chapters, 1,533 verses) using a free **Kaggle GPU (T4 / P100)**.

### Pipeline Steps:
1. **Environment Setup & GPU Check**: Verifies NVIDIA GPU accelerator.
2. **Dependencies**: Installs `ctc-forced-aligner`, `uroman`, `soundfile`, and `imageio-ffmpeg`.
3. **Repo Sync**: Clones/pulls the latest `urhobo-tts` repository from GitHub.
4. **Audio Preparation**: Automatically downloads and conditions chapter audio to 16kHz mono WAV (-23 LUFS, 80Hz HPF).
5. **MMS CTC Alignment**: Aligns audio against romanized transcripts, extracting word-level and verse-level timestamps with confidence scoring.
6. **Export**: Compresses all alignment manifests into `alignments_GEN.zip` for 1-click download.

### Step 1: Verify GPU Environment
Ensure accelerator is set to **GPU T4 x 1** in notebook settings (right panel).

In [ ]:
!nvidia-smi

### Step 2: Install Required Libraries

In [ ]:
!pip install -q git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git uroman soundfile imageio-ffmpeg

### Step 3: Clone & Sync GitHub Repository

In [ ]:
import os
if not os.path.exists('/kaggle/working/urhobo-tts'):
    !git clone https://github.com/ruxy1212/urhobo-tts.git /kaggle/working/urhobo-tts
    %cd /kaggle/working/urhobo-tts
else:
    %cd /kaggle/working/urhobo-tts
    !git pull origin main

### Step 4: Audio Ingestion & Acoustic Conditioning
Downloads raw chapter MP3s (~59 MB total) and applies EBU R128 (-23 LUFS) normalization and 80Hz high-pass filtering (~2 mins total).

In [ ]:
!python scripts/01_scrape_text.py --book GEN --chapters 1-50
!python scripts/02_convert_audio.py --book GEN

### Step 5: Run Forced Alignment (GPU Accelerated)
Runs MMS CTC forced alignment across all 50 chapters of Genesis on CUDA. Estimated runtime: ~10–15 minutes.

In [ ]:
!python scripts/03_align_audio.py --book GEN --device cuda --force

### Step 6: Package & Download Alignments
Creates `alignments_GEN.zip` in `/kaggle/working/` so you can download all 50 alignment JSONs directly.

In [ ]:
!zip -r /kaggle/working/alignments_GEN.zip data/interim/alignments/GEN/
print('\nSUCCESS: alignments_GEN.zip created! Download it from the Kaggle Output section on the right.')